<a href="https://colab.research.google.com/github/sanjushajii/Assignments/blob/main/CVAI_Set_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

np.random.seed(42)

train_path = "/content/drive/MyDrive/Colab Notebooks/train"
test_path = "/content/drive/MyDrive/Colab Notebooks/test"

classes = ["cancer", "tumor", "aneurysm"]

# -------------------------
# DATA STORAGE
# -------------------------
X_train, Y_train = [], []
X_test, Y_test = [], []

# -------------------------
# LOAD TRAIN DATA
# -------------------------
for label, cls in enumerate(classes):

    folder = os.path.join(train_path, cls)

    for file in os.listdir(folder):

        img_path = os.path.join(folder, file)
        img = cv2.imread(img_path)

        if img is None:
            continue

        # Resize
        img = cv2.resize(img, (128, 128))

        # Normalize
        img = img / 255.0

        X_train.append(img)
        Y_train.append(label)

# -------------------------
# LOAD TEST DATA
# -------------------------
for label, cls in enumerate(classes):

    folder = os.path.join(test_path, cls)

    for file in os.listdir(folder):

        img_path = os.path.join(folder, file)
        img = cv2.imread(img_path)

        if img is None:
            continue

        # Resize
        img = cv2.resize(img, (128, 128))

        # Normalize
        img = img / 255.0

        X_test.append(img)
        Y_test.append(label)

# Convert to arrays
X_train = np.array(X_train)
Y_train = np.array(Y_train)
X_test = np.array(X_test)
Y_test = np.array(Y_test)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# -------------------------
# TRAIN-VALIDATION SPLIT
# -------------------------
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train,
    Y_train,
    test_size=0.2,
    random_state=42,
    stratify=Y_train
)

print("Validation shape:", X_val.shape)

# -------------------------
# DATA AUGMENTATION
# -------------------------
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator()

# Fit generator (optional but safe for consistency)
train_datagen.fit(X_train)

Train shape: (2590, 128, 128, 3)
Test shape: (144, 128, 128, 3)
Validation shape: (518, 128, 128, 3)


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([

    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),

    layers.Dense(3, activation='softmax')  # 3 classes
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,027 (12.61 MB)

 Trainable params: 3,305,027 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train model
history = model.fit(
    X_train,
    Y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=8
)

# Save model
model.save("medical_classifier.h5")

print("Model training completed and saved.")

Epoch 1/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 67s 313ms/step - accuracy: 0.9517 - loss: 0.1311 - val_accuracy: 0.9928 - val_loss: 0.0301
Epoch 2/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 64s 308ms/step - accuracy: 0.9861 - loss: 0.0401 - val_accuracy: 0.9928 - val_loss: 0.0205
Epoch 3/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 66s 318ms/step - accuracy: 0.9891 - loss: 0.0290 - val_accuracy: 0.9928 - val_loss: 0.0184
Epoch 4/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 64s 308ms/step - accuracy: 0.9916 - loss: 0.0199 - val_accuracy: 0.9952 - val_loss: 0.0166
Epoch 5/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 66s 315ms/step - accuracy: 0.9916 - loss: 0.0187 - val_accuracy: 0.9928 - val_loss: 0.0532
Epoch 6/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 81s 308ms/step - accuracy: 0.9897 - loss: 0.0209 - val_accuracy: 0.9928 - val_loss: 0.0202
Epoch 7/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 80s 302ms/step - accuracy: 0.9928 - loss: 0.0151 - val_accuracy: 0.9928 - val_loss: 0.0212
Epoch 8/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 62s 300ms/step - accuracy: 0.9916 - loss: 0

Model training completed and saved.


In [5]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Predictions
pred = model.predict(X_test)
pred_classes = np.argmax(pred, axis=1)

# Accuracy
loss, acc = model.evaluate(X_test, Y_test)
print("Test Accuracy:", acc)

# Confusion Matrix
cm = confusion_matrix(Y_test, pred_classes)
print("Confusion Matrix:\n", cm)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(Y_test, pred_classes, target_names=classes))

5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 554ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 440ms/step - accuracy: 1.0000 - loss: 1.3265e-04
Test Accuracy: 1.0
Confusion Matrix:
 [[48  0  0]
 [ 0 48  0]
 [ 0  0 48]]

Classification Report:

              precision    recall  f1-score   support

      cancer       1.00      1.00      1.00        48
       tumor       1.00      1.00      1.00        48
    aneurysm       1.00      1.00      1.00        48

    accuracy                           1.00       144
   macro avg       1.00      1.00      1.00       144
weighted avg       1.00      1.00      1.00       144

